# Create GDX for GAMS Model

Given a year, we create data to be used by the GAMS model

Components that are ready for use in the model:
- capacities from parse_capacities
- prices from parse_prices_entsoe_sftp
- gas prices from parse_gas_prices_eurostat
- load from parse_load_entsoe_sftp and parse_load_eurostat for scaling (CH is assigned scaling of 1 since it is missing from Eurostat)
- NTC values from parse_transfer_capacity, using the TYNDP reference grid 2030
- annual and hourly generation from parse_generation_Eurostat and parse_generation_entsoe_sftp
    - Eurostat is missing data for some countries and technologies so we replace them with ENTSO-E TP values
- hourly RES generation from parse_res_ninja (profiles) generation above
- hourly ror generation from parse_hydro_JRC
- storage and pump_storage inflows from parse_hydro_JRC as well as reservoir inflows calculated based on parse_entsoe_reservoir_level_sftp and parse_generation_entsoe_sftp since Norway 2017 values are missing from the prior source
- hydro reservoir levels from parse_reservoir_level_entsoe_sftp and supplemented with technology shares from parse_hydro_JRC(not using JRC levels from parse_hydro_ENTSO-E as they are identical for all years and only given for a few countries), countries that do not have a value assigned, i.e. start storage is going to be 0 but they get natural inflow over the year: 'BE', 'CZ', 'DE', 'DK', 'EE', 'GB', 'HU', 'IE', 'LU', 'NL', 'PL', 'SK'
- cost resource curves from Cost_resource_curves
- CHP values from parse_CHP_data
- monthly availability of peakload units based on entsoe outage data from parse_outage_entsoe_sftp

Also creates a gdx for calibration with the following parameters:
- r_price(country,t)
- r_demand(country,t)
- r_generation(tech,country,t)
- TRADE(fromcountry,tocountry,t)

Future projects
- load: IEA (https://www.iea.org/reports/monthly-electricity-statistics) seems to have different monthly data than Eurostat, but only for some of our countries - however yearly sums are identical
- load: there are still a few na values for some countries in some hours - so far only backward fill for previous hour to not mess up weekly profiles

## Packages and options

In [2]:
import pandas as pd
import numpy as np
# import gdxtools as gt
# import gams
import os
import sys
import datetime
import calendar
# import qgrid

** Select base year ** 

In [3]:
baseyear = 2024

** Select countries (central or EU)** 

In [4]:
#countries_list = "Countries"
countries_list = "Countries_EU"

In [5]:
if countries_list == "Countries":
    countries_out = ""
if countries_list == "Countries_EU":
    countries_out = "_EU"
countries_out

'_EU'

In [6]:
#file path assignments
fn_additional = "../additional_data.xlsx"
fn_price_gas = "../parsed_data/price_gas_yearly_Eurostat.csv"
fn_cap = "../parsed_data/capacities.csv"
fn_price = "../parsed_data/prices_"+str(baseyear)+"_hourly_entsoe.csv"
fn_load_entsoe = "../parsed_data/load_"+str(baseyear)+"_hourly_entsoe.csv"
fn_load_eurostat = "../parsed_data/load_yearly_Eurostat.csv"
fn_ntc = "../parsed_data/ntc.csv"
fn_trade = "../parsed_data/trade_"+str(baseyear)+"_hourly_entsoe.csv"
fn_res_ninja = "../parsed_data/res_ninja_profiles.csv"
fn_gen_eurostat = "../parsed_data/generation_monthly_eurostat.csv"

fn_gen_entsoe_weekly = '../parsed_data/generation_'+str(baseyear)+'_weekly_entsoe.csv'
fn_gen_entsoe_hourly = '../parsed_data/generation_'+str(baseyear)+'_hourly_entsoe.csv'
fn_avail_peak = '../parsed_data/avail_peakload_GU_'+str(baseyear)+'_monthly_entsoe.csv'

fn_cost_res = "../parsed_data/cost_resource_curves.csv"

#fn_ror = "../parsed_data/hydro_ror_generation_hourly_ENTSO-E_adequacy.csv"
#fn_reservoir_inflow = "../parsed_data/hydro_storage_inflows_weekly_ENTSO-E_adequacy.csv"
fn_reservoir_profile_TP = '../parsed_data/reservoir_level_'+str(baseyear)+'_hourly_entsoe_TP.csv'
fn_reservoir_profile_weekly_TP = '../parsed_data/reservoir_level_'+str(baseyear)+'_weekly_entsoe_TP.csv'
fn_reservoir_size = '../parsed_data/hydro_capacities_base_ENTSO-E_adequacy.csv'

fn_heat_dem = "../parsed_data/heat_demand.csv"
fn_chp_gen = "../parsed_data/chp_generation.csv"

#dir_gdx = "../../"
# we export directly to the models data folder
dir_gdx = os.path.normpath(os.getcwd() + os.sep + os.pardir+ os.sep + os.pardir) + os.sep + "model" + os.sep + "data"  + os.sep
dir_out = "../"

## Additional data and settings

Get additional from Excel and impose settings.

### Sets

Countries in model

In [7]:
df_countries= pd.read_excel(fn_additional, sheet_name=countries_list, index_col="Country")
countries = list(df_countries.index)

Trading partners for demand correction

In [8]:
df_countries_trade = pd.read_excel(fn_additional, sheet_name='Countries_EU_trade', index_col="Country")
countries_trade = list(df_countries_trade.index)

In [9]:
countries_EU_27 = list(pd.read_excel(fn_additional, sheet_name='Countries_EU_27', index_col="Country").index)

Technology sets

In [10]:
df_techs = pd.read_excel(fn_additional, sheet_name="Technologies")
storages = list(df_techs.Hydro.dropna().unique())
renewables = list(df_techs.Renewable.dropna().unique())
old_renewables = list(df_techs["Old Renewables"].dropna().unique())
all_renewables = renewables + old_renewables
conventionals = list(df_techs.Conventional.ffill().unique())
baseload = list(df_techs["Baseload"].dropna().unique())
fixed_feedin = list(df_techs["Fixed"].dropna().unique())
peakload = list(df_techs["peak"].dropna().unique())
technologies = storages + renewables + conventionals

Fuels

In [11]:
df_fuels = pd.read_excel(fn_additional, sheet_name="Fuels")
fuels = list(df_fuels["Main Fuel"].unique())
fuels = [fuel for fuel in fuels if pd.notnull(fuel)]
print(fuels)


['Uran', 'Lignite', 'HardCoal', 'Gas', 'Oil', 'Biomass', 'Hydro', 'Other']


Add fuel cost per country as a separate data frame. 

In [12]:
df_fuels_2017 = pd.read_excel(fn_additional, sheet_name="Fuels_2017")
df_fuels_2017 = df_fuels_2017.fillna(0)
df_fuels_2017.tail()

,country,Main Fuel,Price,Price_2
116,PT,Other,7.0,0.000001
117,RO,Other,7.0,0.000001
118,SE,Other,7.0,0.000001
119,SI,Other,7.0,0.000001
120,SK,Other,7.0,0.000001


We load separate gas prices

In [13]:
df_price_gas = pd.read_csv(fn_price_gas).set_index(['country','year'])
df_price_gas = df_price_gas[df_price_gas.index.get_level_values('year') == baseyear]
df_price_gas = df_price_gas.reset_index().drop(columns='year')
df_price_gas['Main Fuel'] = 'Gas'
df_price_gas.head()

,country,EUR_per_MWh,Main Fuel
0,AT,51.85,Gas
1,BE,42.40,Gas
2,BG,38.10,Gas
3,CZ,61.00,Gas
4,DE,59.00,Gas


In [14]:
df_fuels_2017 = df_fuels_2017.merge(df_price_gas,how='left',left_on=['country','Main Fuel'], right_on=['country','Main Fuel'])
df_fuels_2017 = df_fuels_2017[df_fuels_2017.country.isin(countries)]
df_fuels_2017.loc[df_fuels_2017['EUR_per_MWh'] > 0, 'Price'] = df_fuels_2017['EUR_per_MWh']
df_fuels_2017.tail()

,country,Main Fuel,Price,Price_2,EUR_per_MWh
116,PT,Other,7.0,0.000001,NaN
117,RO,Other,7.0,0.000001,NaN
118,SE,Other,7.0,0.000001,NaN
119,SI,Other,7.0,0.000001,NaN
120,SK,Other,7.0,0.000001,NaN


Add variable O&M cost as a separate data frame:

In [15]:
df_OM_2017 = pd.read_excel(fn_additional, sheet_name="OM_2017")
df_OM_2017.head(1)

,country,Technology,variable_OM_Cost
0,AT,Biomass,2.6


### Mappings for missing data

Mapping for missing fuel prices

In [16]:
df_fuel_price_maps = pd.read_excel(fn_additional, sheet_name="FuelPriceAs", index_col = 0)

fuel_price_maps = []
for row in df_fuel_price_maps.iterrows():
    country = row[0]
    for fuel in df_fuel_price_maps.columns:
        fuel_price_as = row[1].loc[fuel]
        if pd.notnull(fuel_price_as):
            fuel_price_maps.append((country, fuel_price_as, fuel))

Mapping for missing variable O&M costs

In [17]:
df_om_maps = pd.read_excel(fn_additional, sheet_name="OMAs", index_col = 0)

om_maps = []
for row in df_om_maps.iterrows():
    country = row[0]
    for tech in df_om_maps.columns:
        om_as = row[1].loc[tech]
        if pd.notnull(om_as):
            om_maps.append((country, om_as, tech))

### Technology cost specifications

In [18]:
df_cost_in = pd.read_excel(fn_additional, sheet_name="Cost")
df_cost_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Main Fuel                                12 non-null     object 
 1   Technology                               15 non-null     object 
 2   Average Efficiency                       15 non-null     float64
 3   Availability                             15 non-null     float64
 4   Min Generation                           15 non-null     float64
 5   Startup Costs                            15 non-null     int64  
 6   Reserve                                  15 non-null     int64  
 7   variable OM Cost [Euro/MWh]              15 non-null     float64
 8   variable OM Cost quadratic [Euro/MWh^2]  15 non-null     float64
 9   LoadGradient                             15 non-null     float64
 10  RampingCost                              15 non-null

In [19]:
lst_df = []
for c in countries:
    df = df_cost_in.copy()
    df["country"] = c
    lst_df.append(df)
df_cost = pd.concat(lst_df, sort=True)

Additional availability data for peak load technologies based on entsoe outage data

In [20]:
df_avail_peak = pd.read_csv(fn_avail_peak).set_index(['country','technology','Month'])
df_avail_peak = df_avail_peak[df_avail_peak.index.get_level_values('country').isin(countries)]
df_avail_peak = df_avail_peak[df_avail_peak.index.get_level_values('technology').isin(peakload)]
df_avail_peak.head()

avail
country technology Month          
AT      Gas        1      0.700262
                   2      0.575143
                   3      0.374085
                   4      0.188551
                   5      0.104083

## Capacities

Select country and year and make unique plants prefixing technologies by country name:

In [21]:
df_cap_in = pd.read_csv(fn_cap)
df_cap = df_cap_in[(df_cap_in.country.isin(countries)
                    & df_cap_in.technology.isin(technologies)
                    )]
#df_cap = df_cap[df_cap.year == baseyear].drop('year',axis=1).set_index(['country','technology'])
#this does not seem to be used in final code but lets not delete it for now
#df_cap["plant"] = df_cap.country + "_" + df_cap.technology
df_cap.head()

,country,technology,capacity
13,AT,Biomass,549.0
14,AT,Gas,4225.0
15,AT,HardCoal,0.0
16,AT,Lignite,0.0
17,AT,Nuclear,0.0


we load capacities from the new hydro data set

even though the baseyear is not given for ENTSO-E capacities, they seem to be fairly recent and much more comprehensive (covering more countries) than what we had before so we just use them

In [22]:
df_cap_hydro = pd.read_csv(fn_reservoir_size)
df_cap_hydro = df_cap_hydro.rename(columns={
    'Pump Storage - Closed Loop - Total turbining capacity (MW)':'PumpClosed',  
    'Pump Storage - Open Loop - Total turbining capacity (MW)':'PumpOpen',
#    'Reservoir - Total turbining capacity (MW)':'Reservoir',
#    'Run-of-River and pondage - Total turbining capacity (MW)':'RunOfRiver'
                                }).set_index('country')
df_cap_hydro = df_cap_hydro[['PumpClosed','PumpOpen']]
df_cap_hydro = pd.DataFrame(df_cap_hydro.stack()).reset_index()
df_cap_hydro.columns = ['country','technology','capacity']
df_cap_hydro = pd.DataFrame(df_cap_hydro.set_index(['country','technology'])['capacity'])
df_cap_hydro.head()

capacity
country technology          
AL      PumpClosed      0.00
        PumpOpen        0.00
AT      PumpClosed      0.00
        PumpOpen     3459.28
BA      PumpClosed      0.00

load pumping capacity as well and merge the two

In [23]:
df_cap_hydro_pump = pd.read_csv(fn_reservoir_size)
df_cap_hydro_pump = df_cap_hydro_pump.rename(columns={
    'Pump Storage - Closed Loop - Total pumping capacity (MW)':'PumpClosed',  
    'Pump Storage - Open Loop - Total pumping capacity (MW)':'PumpOpen'}
                                 ).set_index('country')
df_cap_hydro_pump = df_cap_hydro_pump[['PumpClosed','PumpOpen']]
df_cap_hydro_pump = pd.DataFrame(df_cap_hydro_pump.stack()).reset_index()
df_cap_hydro_pump.columns = ['country','technology','pumping']
df_cap_hydro_pump = pd.DataFrame(df_cap_hydro_pump.set_index(['country','technology'])['pumping'])
df_cap_hydro_pump = abs(df_cap_hydro_pump)
df_cap_hydro = df_cap_hydro.join(df_cap_hydro_pump,how='outer')
df_cap_hydro = df_cap_hydro.reset_index()
df_cap_hydro.head()

,country,technology,capacity,pumping
0,AL,PumpClosed,0.00,0.000
1,AL,PumpOpen,0.00,0.000
2,AT,PumpClosed,0.00,0.000
3,AT,PumpOpen,3459.28,2559.726
4,BA,PumpClosed,0.00,0.000


now join with entsoe

In [24]:
df_cap = pd.concat([df_cap, df_cap_hydro])
df_cap = df_cap.reset_index()
df_cap = df_cap[(df_cap.country.isin(countries)
                    & df_cap.technology.isin(technologies)
                    )].drop(columns='index')
df_cap = df_cap[df_cap['technology'] != 'Pump']
df_cap = df_cap.set_index(['technology','country']).fillna(0)
df_cap = abs(df_cap)
df_cap.head()

,,capacity,pumping
technology,country,,
Biomass,AT,549.0,0.0
Gas,AT,4225.0,0.0
HardCoal,AT,0.0,0.0
Lignite,AT,0.0,0.0
Nuclear,AT,0.0,0.0


## Generation for all technologies

### generation eurostat and entsoe

load monthly values from eurostat

In [25]:
df_monthly_generation_in =  pd.read_csv(fn_gen_eurostat)
df_monthly_generation_in = df_monthly_generation_in[df_monthly_generation_in.year == baseyear]
df_monthly_generation = df_monthly_generation_in[df_monthly_generation_in.country.isin(countries)]
df_monthly_generation = df_monthly_generation.rename(columns={'tech':'technology'})
df_monthly_generation = df_monthly_generation[df_monthly_generation['MWh'] > 0]
df_monthly_generation.head(1)

,country,time,technology,year,month,MWh
2509,AT,2024-01,Biomass,2024,1,241363.0


create yearly eurostat df

In [26]:
df_eurostat_year = df_monthly_generation.groupby(['year','country','technology']).sum().reset_index().drop(['time','year','month'], axis=1).set_index(['country','technology'])
df_eurostat_year = df_eurostat_year.rename(columns={'MWh':'eurostat'})
df_eurostat_year.head(1)

,,eurostat
country,technology,
AT,Biomass,2672337.0


also load entsoe TP hourly data

In [27]:
df_hourly_generation_entsoe_in =  pd.read_csv(fn_gen_entsoe_hourly,parse_dates=True, index_col="date").reset_index()
df_hourly_entsoe = df_hourly_generation_entsoe_in.rename(columns={'tech':'technology','date':'time','net_generation':'MWh_hourly'})
df_hourly_entsoe = df_hourly_entsoe.set_index(['country','technology','time'])[['MWh_hourly']]
df_hourly_entsoe.head(1)

,,,MWh_hourly
country,technology,time,
AT,Biomass,2024-01-01,143.0


In [28]:
df_hourly_entsoe_test = df_hourly_entsoe.copy()
df_hourly_entsoe_test['generator'] = df_hourly_entsoe_test.index.get_level_values('country') + df_hourly_entsoe_test.index.get_level_values('technology')
df_hourly_entsoe_test = df_hourly_entsoe_test.groupby('generator').sum()
df_hourly_entsoe_test = df_hourly_entsoe_test[df_hourly_entsoe_test.MWh_hourly > 0]
df_hourly_entsoe_test.head(1)

,MWh_hourly
generator,
ATBiomass,1933479.0


check for missing values in entsoe

In [29]:
df_eurostat_year_test = df_eurostat_year[df_eurostat_year.index.get_level_values('technology').isin(technologies)].copy()
df_eurostat_year_test['generator'] = df_eurostat_year_test.index.get_level_values('country') + df_eurostat_year_test.index.get_level_values('technology')

In [30]:
missing = np.setdiff1d(df_eurostat_year_test.generator.unique(),df_hourly_entsoe_test.index.get_level_values('generator').unique())
df_eurostat_year_test.eurostat[df_eurostat_year_test.generator.isin(missing)].sum() / df_eurostat_year_test.eurostat.sum() * 100
print(missing)

['ATOil' 'BEPump' 'BGOil' 'CZPump' 'DEPump' 'ESPump' 'FIWindOffshore'
 'IEBiomass' 'IEPump' 'IESolar' 'LUPump' 'NLOil' 'NOBiomass' 'PTOil'
 'ROOil' 'SEBiomass' 'SEOil' 'SEWindOffshore']


In [31]:
missing

array(['ATOil', 'BEPump', 'BGOil', 'CZPump', 'DEPump', 'ESPump',
       'FIWindOffshore', 'IEBiomass', 'IEPump', 'IESolar', 'LUPump',
       'NLOil', 'NOBiomass', 'PTOil', 'ROOil', 'SEBiomass', 'SEOil',
       'SEWindOffshore'], dtype=object)

In [32]:
df_eurostat_year_test.head()

eurostat      generator
country technology                            
AT      Biomass       2672337.0      ATBiomass
        Gas           6892224.0          ATGas
        Oil            594298.0          ATOil
        Other        11010533.0        ATOther
        WindOnshore   9134588.0  ATWindOnshore

We manually fill missing thermal generation profiles with profiles from neighbouring countries, aggregate values will be set to eurostat

In [33]:
df_hourly_entsoe_pivot = df_hourly_entsoe.pivot_table(index=['time','technology'],columns='country',values='MWh_hourly')

df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'AT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BG'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'IT']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'IE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'GB']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'NL'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'NO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'PT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'ES']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'RO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'GR']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DK']

df_hourly_entsoe_filled = pd.DataFrame(df_hourly_entsoe_pivot.stack()).rename(columns={0:'MWh_hourly'}).reset_index().set_index(['country','technology','time'])
df_hourly_entsoe_filled.head(1)

,,,MWh_hourly
country,technology,time,
AT,Biomass,2024-01-01,143.0


create yearly aggregate

In [34]:
df_yearly_entsoe = df_hourly_entsoe_filled.reset_index().drop('time', axis=1).groupby(['country','technology']).sum()
df_yearly_entsoe = df_yearly_entsoe.rename(columns={'MWh_hourly':'entsoe'})
df_yearly_entsoe.head()

entsoe
country technology              
AT      Biomass     1.933479e+06
        Gas         6.149846e+06
        HardCoal    0.000000e+00
        Oil         3.243173e+06
        Other       1.072263e+06

merge the two yearly dfs and fill missing (or smaller) eurostat data with entsoe values 

In [35]:
df_generation_year = df_yearly_entsoe.join(df_eurostat_year, how='outer')
df_generation_year['MWh_year'] = df_generation_year['eurostat'].fillna(df_generation_year['entsoe'])
df_generation_year.loc[df_generation_year['entsoe'] > df_generation_year['eurostat'], 'MWh_year'] = df_generation_year['entsoe']
#replace 0s from Eurostat with NaN, otherwise, we get inf values for scaling
df_generation_year['MWh_year'] = df_generation_year['MWh_year'].replace(0, np.nan)
df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     1933479.0   2672337.0   2672337.0
        Coal              NaN   1798217.0   1798217.0
        Gas         6149845.6   6892224.0   6892224.0
        HardCoal          0.0         NaN         NaN
        Hydro             NaN  44450305.0  44450305.0

In [36]:
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'AT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BG'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'IT']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'IE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'GB']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'NL'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'BE']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'NO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'PT'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'ES']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'RO'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'GR']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Biomass', 'DK']
df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'SE'] = df_hourly_entsoe_pivot.loc[df_hourly_entsoe_pivot.index.get_level_values('technology') == 'Oil', 'DK']


Make sure that filled values doe not mess with scaling

In [37]:
df_generation_year.loc[('AT', 'Oil'), 'MWh_year'] = df_generation_year.loc[('AT', 'Oil'), 'eurostat']
df_generation_year.loc[('BG', 'Oil'), 'MWh_year'] = df_generation_year.loc[('BG', 'Oil'), 'eurostat']
df_generation_year.loc[('IE', 'Biomass'), 'MWh_year'] = df_generation_year.loc[('IE', 'Biomass'), 'eurostat']
df_generation_year.loc[('NL', 'Oil'), 'MWh_year'] = df_generation_year.loc[('NL', 'Oil'), 'eurostat']
df_generation_year.loc[('NO', 'Biomass'), 'MWh_year'] = df_generation_year.loc[('NO', 'Biomass'), 'eurostat']
df_generation_year.loc[('PT', 'Oil'), 'MWh_year'] = df_generation_year.loc[('PT', 'Oil'), 'eurostat']
df_generation_year.loc[('RO', 'Oil'), 'MWh_year'] = df_generation_year.loc[('RO', 'Oil'), 'eurostat']
df_generation_year.loc[('SE', 'Biomass'), 'MWh_year'] = df_generation_year.loc[('SE', 'Biomass'), 'eurostat']
df_generation_year.loc[('SE', 'Oil'), 'MWh_year'] = df_generation_year.loc[('SE', 'Oil'), 'eurostat']

df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     1933479.0   2672337.0   2672337.0
        Coal              NaN   1798217.0   1798217.0
        Gas         6149845.6   6892224.0   6892224.0
        HardCoal          0.0         NaN         NaN
        Hydro             NaN  44450305.0  44450305.0

Eurostat only has coal but we can scale values in countries where only one of the two is present

In [38]:
df_generation_year.loc[('AT', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('AT', 'Coal'), 'eurostat']
df_generation_year.loc[('DK', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('DK', 'Coal'), 'eurostat']
df_generation_year.loc[('FI', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('FI', 'Coal'), 'eurostat']
df_generation_year.loc[('FR', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('FR', 'Coal'), 'eurostat']
df_generation_year.loc[('NL', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('NL', 'Coal'), 'eurostat']
df_generation_year.loc[('SE', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('SE', 'Coal'), 'eurostat']

df_generation_year.loc[('HR', 'HardCoal'), 'MWh_year'] = df_generation_year.loc[('HR', 'Coal'), 'eurostat']
df_generation_year.head()

entsoe    eurostat    MWh_year
country technology                                   
AT      Biomass     1933479.0   2672337.0   2672337.0
        Coal              NaN   1798217.0   1798217.0
        Gas         6149845.6   6892224.0   6892224.0
        HardCoal          0.0         NaN   1798217.0
        Hydro             NaN  44450305.0  44450305.0

Eventually, calculate scaling and drop not needed techs

In [39]:
df_generation_year['scale'] = df_generation_year['MWh_year']/df_generation_year['entsoe']
df_generation_year = df_generation_year[df_generation_year.index.get_level_values('technology').isin(technologies)]

In [40]:
df_generation_year.head()

entsoe    eurostat    MWh_year      scale
country technology                                                 
AT      Biomass     1.933479e+06   2672337.0   2672337.0   1.382139
        Gas         6.149846e+06   6892224.0   6892224.0   1.120715
        HardCoal    0.000000e+00         NaN   1798217.0        inf
        Oil         3.243173e+06    594298.0    594298.0   0.183246
        Other       1.072263e+06  11010533.0  11010533.0  10.268502

Use calculated scaling to scale hourly entsoe values 

In [41]:
df_generation_hourly_1 = pd.DataFrame()
df_generation_hourly_1 = df_hourly_entsoe_filled.join(df_generation_year[['scale','MWh_year']],how='outer')
df_generation_hourly_1['MWh'] = df_generation_hourly_1['MWh_hourly'] * df_generation_hourly_1['scale']
df_generation_hourly_1 = df_generation_hourly_1[['MWh','MWh_year']]
df_generation_hourly_1.head()

MWh   MWh_year
country technology time                                      
AT      Biomass    2024-01-01 00:00:00  197.645897  2672337.0
                   2024-01-01 01:00:00  193.499479  2672337.0
                   2024-01-01 02:00:00  193.499479  2672337.0
                   2024-01-01 03:00:00  193.499479  2672337.0
                   2024-01-01 04:00:00  193.499479  2672337.0

### Renewable generation

Get renewable generation for countries and base year

In [42]:
df_res_ninja_in = pd.read_csv(fn_res_ninja, parse_dates=True, index_col="time")
df_res_ninja = df_res_ninja_in.loc[
    (df_res_ninja_in['country'].isin(countries)) &
    (df_res_ninja_in['tech'].isin(renewables)) &
    (df_res_ninja_in.index.year == baseyear)
].copy().reset_index()

In [43]:
df_res_ninja_in.tail()

,country,tech,capacity_factor
time,,,
2024-12-31 23:00:00+00:00,PT,WindOffshore,0.620390
2024-12-31 23:00:00+00:00,RO,WindOffshore,0.232654
2024-12-31 23:00:00+00:00,SE,WindOffshore,0.607949
2024-12-31 23:00:00+00:00,SI,WindOffshore,0.000000
2024-12-31 23:00:00+00:00,SK,WindOffshore,0.000000


In [44]:
df_res_ninja_in.loc[
    (df_res_ninja_in['country'].isin(countries)) &
    (df_res_ninja_in['tech'].isin(renewables)) &
    (df_res_ninja_in.index.year == 2024)
].head()

,country,tech,capacity_factor
time,,,
2024-01-01 00:00:00+00:00,AT,Solar,0.0
2024-01-01 00:00:00+00:00,BE,Solar,0.0
2024-01-01 00:00:00+00:00,BG,Solar,0.0
2024-01-01 00:00:00+00:00,CH,Solar,0.0
2024-01-01 00:00:00+00:00,CZ,Solar,0.0


We normalize to hourly share of total yearly generation

In [45]:
df_res_ninja_year = df_res_ninja.reset_index().drop(columns='time').groupby(['country','tech']).sum()
df_res_ninja_profile = df_res_ninja.merge(df_res_ninja_year,how='left',on=['country','tech'])
df_res_ninja_profile['profile'] = df_res_ninja_profile['capacity_factor_x']/df_res_ninja_profile['capacity_factor_y']
df_res_ninja_profile = df_res_ninja_profile.rename(columns={'tech':'technology'})
df_res = df_res_ninja_profile.set_index(['country','technology','time'])[['profile']]
df_res.head()


,,,profile
country,technology,time,
AT,Solar,2024-01-01 00:00:00+00:00,0.0
BE,Solar,2024-01-01 00:00:00+00:00,0.0
BG,Solar,2024-01-01 00:00:00+00:00,0.0
CH,Solar,2024-01-01 00:00:00+00:00,0.0
CZ,Solar,2024-01-01 00:00:00+00:00,0.0


now we add the profile to hourly generation

In [46]:
df_generation_hourly_2 = df_generation_hourly_1.join(df_res, how='outer')
df_generation_hourly_2['MWh'][df_generation_hourly_2['profile'].notna()] = df_generation_hourly_2['MWh_year'] * df_generation_hourly_2['profile']
df_generation_hourly_2.head()

C:\Users\jonas\AppData\Local\Temp\ipykernel_45696\3190696675.py:1: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  df_generation_hourly_2 = df_generation_hourly_1.join(df_res, how='outer')
C:\Users\jonas\AppData\Local\Temp\ipykernel_45696\3190696675.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the doc

MWh   MWh_year  profile
country technology time                                               
AT      Biomass    2024-01-01 00:00:00  197.645897  2672337.0      NaN
                   2024-01-01 01:00:00  193.499479  2672337.0      NaN
                   2024-01-01 02:00:00  193.499479  2672337.0      NaN
                   2024-01-01 03:00:00  193.499479  2672337.0      NaN
                   2024-01-01 04:00:00  193.499479  2672337.0      NaN

### merging all generation dfs

In [47]:
df_generation = df_generation_hourly_2.copy() #skips now run of river from JRC
df_generation = df_generation[['MWh','profile']].fillna(0)
df_generation = df_generation[df_generation.index.get_level_values('country').isin(countries)]
df_generation.head(1)

,,,MWh,profile
country,technology,time,,
AT,Biomass,2024-01-01 00:00:00,197.645897,0.0


Also create hourly res and ror dfs for easier export

In [48]:
df_res = df_generation.copy()
df_res = df_res[df_res.index.get_level_values('technology').isin(renewables)]
df_res.head(20)

MWh  profile
country technology time                               
AT      Solar      2024-01-01 00:00:00    0.0      0.0
                   2024-01-01 01:00:00    0.0      0.0
                   2024-01-01 02:00:00    0.0      0.0
                   2024-01-01 03:00:00    0.0      0.0
                   2024-01-01 04:00:00    0.0      0.0
                   2024-01-01 05:00:00    0.0      0.0
                   2024-01-01 06:00:00    9.0      0.0
                   2024-01-01 07:00:00   85.0      0.0
                   2024-01-01 08:00:00  221.0      0.0
                   2024-01-01 09:00:00  474.0      0.0
                   2024-01-01 10:00:00  683.0      0.0
                   2024-01-01 11:00:00  753.0      0.0
                   2024-01-01 12:00:00  670.0      0.0
                   2024-01-01 13:00:00  469.0      0.0
                   2024-01-01 14:00:00  214.0      0.0
                   2024-01-01 15:00:00   21.0      0.0
                   2024-01-01 16:00:00    0.0      0.0
                   2024-01-01 17:00:00    0.0      0.0
                   2024-01-01 18:00:00    0.0      0.0
                   2024-01-01 19:00:00    0.0      0.0

In [49]:
df_ror = df_generation.reset_index().copy()
df_ror = df_ror[df_ror.technology.isin({'RunOfRiver'})].drop(columns=['technology'])
df_ror.loc[df_ror['MWh'] < 0, 'MWh'] = 0
df_ror.head()

,country,time,MWh,profile
61488,AT,2024-01-01 00:00:00,4085.3,0.0
61489,AT,2024-01-01 01:00:00,3999.2,0.0
61490,AT,2024-01-01 02:00:00,3836.3,0.0
61491,AT,2024-01-01 03:00:00,3831.4,0.0
61492,AT,2024-01-01 04:00:00,3976.4,0.0


Drop profile from df_generation

In [50]:
df_generation = df_generation[['MWh']]
df_generation.head()

MWh
country technology time                           
AT      Biomass    2024-01-01 00:00:00  197.645897
                   2024-01-01 01:00:00  193.499479
                   2024-01-01 02:00:00  193.499479
                   2024-01-01 03:00:00  193.499479
                   2024-01-01 04:00:00  193.499479

Create monthly generation df for availability calibration of baseload technologies

In [51]:
df_generation.reset_index().time.tail()

2775611    2024-12-31 23:00:00+00:00
2775612    2024-12-31 23:00:00+00:00
2775613    2024-12-31 23:00:00+00:00
2775614    2024-12-31 23:00:00+00:00
2775615    2024-12-31 23:00:00+00:00
Name: time, dtype: object

In [52]:
df_generation = df_generation.reset_index()
df_generation['time'] = pd.to_datetime(df_generation['time'], utc=True)
df_generation = df_generation.set_index(['country', 'technology', 'time'])[['MWh']]

In [53]:
#JONAS add pump shares here
df_generation.index.get_level_values('technology').unique()

Index(['Biomass', 'Gas', 'HardCoal', 'Oil', 'Other', 'Pump', 'Reservoir',
       'RunOfRiver', 'Solar', 'WindOnshore', 'Nuclear', 'WindOffshore',
       'Lignite'],
      dtype='object', name='technology')

In [54]:
df_generation_monthly = df_generation.groupby([pd.Grouper(level='time', freq='ME'),
                                               'country', 'technology']).sum().reset_index()
df_generation_monthly['month'] = df_generation_monthly['time'].dt.month
df_generation_monthly = df_generation_monthly.set_index(['month', 'country', 'technology'])[['MWh']]
df_generation_monthly.head()

MWh
month country technology              
1     AT      Biomass     1.671103e+05
              Gas         1.354625e+06
              HardCoal    0.000000e+00
              Oil         5.391680e+04
              Other       9.325861e+05

In [55]:
#check for missing countries
missing = np.setdiff1d(countries,np.unique(df_generation_monthly.reset_index().country.unique()))
missing

array([], dtype='<U2')

In [56]:
df_gen_annual = df_generation_monthly.groupby(['country','technology']).sum()

## Load

Get load and select countries and baseyear.

In [57]:
df_load_in_entsoe = pd.read_csv(fn_load_entsoe, parse_dates=True, index_col="date")[countries].copy()
df_load_in_eurostat = pd.read_csv(fn_load_eurostat, index_col="country")[str(baseyear)].copy()
df_load_in_eurostat = pd.DataFrame(df_load_in_eurostat[df_load_in_eurostat.index.isin(countries)])
#we have some missing values which we fill with value from previous hour (up to 10 hours)
df_load_in_entsoe = df_load_in_entsoe.fillna(method = 'pad', limit=10) 
df_load_in_entsoe =df_load_in_entsoe[df_load_in_entsoe.index.year == baseyear]
df_load_in_entsoe.info() #looks ok apart from LT - not too important for now...

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8784 entries, 2024-01-01 00:00:00 to 2024-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8784 non-null   float64
 1   BE      8784 non-null   float64
 2   BG      8784 non-null   float64
 3   HR      8784 non-null   float64
 4   CZ      8784 non-null   float64
 5   DK      8784 non-null   float64
 6   FI      8784 non-null   float64
 7   FR      8784 non-null   float64
 8   DE      8784 non-null   float64
 9   GR      8784 non-null   float64
 10  HU      8784 non-null   float64
 11  IE      8784 non-null   float64
 12  IT      8784 non-null   float64
 13  LU      8784 non-null   float64
 14  NL      8784 non-null   float64
 15  PL      8784 non-null   float64
 16  PT      8784 non-null   float64
 17  RO      8784 non-null   float64
 18  SK      8784 non-null   float64
 19  SI      8784 non-null   float64
 20  ES      8784 non-null   float64
 21  S

C:\Users\jonas\AppData\Local\Temp\ipykernel_45696\4291086698.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_load_in_entsoe = df_load_in_entsoe.fillna(method = 'pad', limit=10)


In [58]:
#we fill the rest with 0s for now
df_load_in_entsoe = df_load_in_entsoe.fillna(0)
df_load_in_entsoe.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8784 entries, 2024-01-01 00:00:00 to 2024-12-31 23:00:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8784 non-null   float64
 1   BE      8784 non-null   float64
 2   BG      8784 non-null   float64
 3   HR      8784 non-null   float64
 4   CZ      8784 non-null   float64
 5   DK      8784 non-null   float64
 6   FI      8784 non-null   float64
 7   FR      8784 non-null   float64
 8   DE      8784 non-null   float64
 9   GR      8784 non-null   float64
 10  HU      8784 non-null   float64
 11  IE      8784 non-null   float64
 12  IT      8784 non-null   float64
 13  LU      8784 non-null   float64
 14  NL      8784 non-null   float64
 15  PL      8784 non-null   float64
 16  PT      8784 non-null   float64
 17  RO      8784 non-null   float64
 18  SK      8784 non-null   float64
 19  SI      8784 non-null   float64
 20  ES      8784 non-null   float64
 21  S

In [59]:
df_load_in_entsoe_year = df_load_in_entsoe.resample('Y').sum().T.reset_index()
df_load_in_entsoe_year.columns = ['country',baseyear]
df_load_in_entsoe_year = df_load_in_entsoe_year.set_index('country')

C:\Users\jonas\AppData\Local\Temp\ipykernel_45696\3099131928.py:1: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  df_load_in_entsoe_year = df_load_in_entsoe.resample('Y').sum().T.reset_index()


calculate and apply weighting based on eurostat yearly values

In [60]:
df_load_weighting = df_load_in_eurostat[str(baseyear)]/df_load_in_entsoe_year[baseyear]
df_load_weighting[df_load_weighting.isna()] = 1 #Switzerland is missing from Eurostat data, so we assign a value of 1
df_load = df_load_in_entsoe * df_load_weighting.T
df_load = pd.DataFrame(df_load.stack()).reset_index().rename(columns=
                                                            {'date':'time','level_1':'country',0:'MWh'}).set_index(['country','time'])
df_load.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 219600 entries, ('AT', Timestamp('2024-01-01 00:00:00')) to ('SK', Timestamp('2024-12-31 23:00:00'))
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   MWh     219600 non-null  float64
dtypes: float64(1)
memory usage: 2.6+ MB


correct load by trade with countries outside of the system

In [61]:
df_trade = pd.read_csv(fn_trade, parse_dates=True, index_col="time")
df_trade.head()

,from_country,to_country,MWh
time,,,
2024-01-01 00:00:00,AL,GR,348.0
2024-01-01 01:00:00,AL,GR,312.0
2024-01-01 02:00:00,AL,GR,261.0
2024-01-01 03:00:00,AL,GR,312.0
2024-01-01 04:00:00,AL,GR,358.0


In [62]:
df_trade_import = df_trade[(df_trade.from_country.isin(countries_trade) & df_trade.to_country.isin(countries))]
df_trade_import = df_trade_import.groupby(['time','to_country']).sum()
df_trade_import = df_trade_import.reset_index().rename(columns={'to_country':'country','MWh':'import'}).set_index(['country','time'])
df_trade_import.head()

,,from_country,import
country,time,,
BG,2024-01-01,MKRSTR,322.0
FI,2024-01-01,EE,0.0
GB,2024-01-01,NI,0.0
GR,2024-01-01,ALMKTR,898.0
HR,2024-01-01,BARS,172.0


In [63]:
df_trade_export = df_trade[(df_trade.from_country.isin(countries) & df_trade.to_country.isin(countries_trade))]
df_trade_export = df_trade_export.groupby(['time','from_country']).sum()
df_trade_export = df_trade_export.reset_index().rename(columns={'from_country':'country','MWh':'export'}).set_index(['country','time'])
df_trade_export.head()

,,to_country,export
country,time,,
BG,2024-01-01,MKRSTR,724.00
FI,2024-01-01,EE,25.70
GB,2024-01-01,NI,116.12
GR,2024-01-01,ALMKTR,93.00
HR,2024-01-01,BARS,1070.00


In [64]:
df_load_corrected = df_load.merge(df_trade_export,how='left',left_index=True,right_index=True)
df_load_corrected = df_load_corrected.merge(df_trade_import,how='left',left_index=True,right_index=True)
df_load_corrected = df_load_corrected.fillna(0)
df_load_corrected['MWh_corrected'] = df_load_corrected['MWh'] + df_load_corrected['export'] - df_load_corrected['import']
df_load_corrected.head()

,,MWh,to_country,export,from_country,import,MWh_corrected
country,time,,,,,,
AT,2024-01-01,7114.960493,0,0.0,0,0.0,7114.960493
BE,2024-01-01,7791.499888,0,0.0,0,0.0,7791.499888
BG,2024-01-01,3719.916535,MKRSTR,724.0,MKRSTR,322.0,4121.916535
CH,2024-01-01,7692.420000,0,0.0,0,0.0,7692.420000
CZ,2024-01-01,5406.481801,0,0.0,0,0.0,5406.481801


In [65]:
df_load = pd.DataFrame()
df_load = df_load_corrected[['MWh_corrected']].copy().rename(columns={'MWh_corrected':'MWh'})
df_load.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 219600 entries, ('AT', Timestamp('2024-01-01 00:00:00')) to ('SK', Timestamp('2024-12-31 23:00:00'))
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   MWh     219600 non-null  float64
dtypes: float64(1)
memory usage: 10.7+ MB


### Rescale demand to fit system generation

In [66]:
#load is higher than generation so we need some scaling in the model
load_scaling = df_generation['MWh'].sum()/df_load['MWh'].sum()
df_load['MWh'] = df_load['MWh'] * load_scaling
df_generation['MWh'].sum()/df_load['MWh'].sum()

0.9999999999999999

In [67]:
load_scaling

0.9470582880840716

## Reservoir inflows

Since JRC only goes to 2017, we switch to ENTSO-E Transparency generation and inflow and distribute them according to per technology share

In [95]:
#we load weekly values as this is best temporal resolution for storage levels
df_reservoir_level_weekly = pd.read_csv(fn_reservoir_profile_weekly_TP,parse_dates=True, index_col="date").reset_index()
df_gen_weekly_in =  pd.read_csv(fn_gen_entsoe_weekly,parse_dates=True, index_col="date").reset_index()

In [99]:
#only keep reservoirs and net_generation, pumping is included here
df_gen_weekly_reservoir = df_gen_weekly_in[df_gen_weekly_in.tech=='Reservoir'][['date','country','net_generation']]
#adjust datetime for gen_weekly as reservoir level is one day later in 2017

df_gen_weekly_reservoir.date = df_gen_weekly_reservoir.date + datetime.timedelta(days=1)
df_gen_weekly_reservoir = df_gen_weekly_reservoir.set_index(['date','country'])
df_gen_weekly_reservoir.head()

net_generation
date       country                
2024-01-08 AT             79770.10
           BA             87830.29
           BG             13930.80
           CH            127770.83
           CZ             87162.21

In [109]:
#create a df with reservoir levels from previus week
df_reservoir_level_weekly_before = df_reservoir_level_weekly.copy()
df_reservoir_level_weekly_before.date = df_reservoir_level_weekly_before.date + datetime.timedelta(weeks=1)
df_reservoir_level_weekly_before = df_reservoir_level_weekly_before.rename(columns = {'MWh':'MWh_t-1'})
df_reservoir_level_weekly_before.head()

,date,country,MWh_t-1
0,2024-01-08,GE,440.0
1,2024-01-08,RS,584000.0
2,2024-01-08,ES,9429185.0
3,2024-01-08,FI,3410560.0
4,2024-01-08,FR,2736874.0


In [110]:
#merge all three dfs and calculate inflows
df_reservoir_level_weekly_merge = df_reservoir_level_weekly.merge(df_reservoir_level_weekly_before,
                                                            how='left',on=['date','country']).dropna()
df_reservoir_level_weekly_merged = df_reservoir_level_weekly_merge.merge(df_gen_weekly_reservoir,
                                                                        how='left',on=['date','country']).dropna()
df_reservoir_level_weekly_merged['natural_inflow'] = df_reservoir_level_weekly_merged['MWh'] - df_reservoir_level_weekly_merged['MWh_t-1'] + df_reservoir_level_weekly_merged['net_generation']
df_reservoir_level_weekly_merged.head()

,date,country,MWh,MWh_t-1,net_generation,natural_inflow
1,2024-01-08,IT,3053817.00,3216474.00,63302.00,-99355.00
2,2024-01-08,FR,2368286.00,2736874.00,394191.00,25603.00
3,2024-01-08,AT,1423062.91,1537770.73,79770.10,-34937.72
4,2024-01-08,HR,774070.00,781990.00,97075.20,89155.20
6,2024-01-08,NO,49076281.00,51945281.00,2710148.28,-158851.72


In [72]:
# Upsample weekly reservoir inflows to hourly values (fixed)
df_reservoir_inflow_entsoe = df_reservoir_level_weekly_merged[['date','country','natural_inflow']].copy()

# replace negative inflows with 0 and convert to hourly rate
df_reservoir_inflow_entsoe['natural_inflow'] = df_reservoir_inflow_entsoe['natural_inflow'].clip(lower=0)
df_reservoir_inflow_entsoe['natural_inflow'] = df_reservoir_inflow_entsoe['natural_inflow'] / (7 * 24)

# add boundary timestamps for all countries so resampling can fill the full range
extras = []
start_ts = pd.Timestamp(f'{baseyear-1}-12-31')
end_ts = pd.Timestamp(f'{baseyear+1}-01-01')
for c in countries:
    extras.append({'date': start_ts, 'country': c, 'natural_inflow': np.nan})
    extras.append({'date': end_ts, 'country': c, 'natural_inflow': np.nan})
if extras:
    df_reservoir_inflow_entsoe = pd.concat([df_reservoir_inflow_entsoe, pd.DataFrame(extras)], ignore_index=True)

# pivot to wide, resample hourly and forward/back fill, then convert back to long
df_wide = df_reservoir_inflow_entsoe.set_index('date').pivot(columns='country', values='natural_inflow')
df_wide = df_wide.sort_index().resample('H').ffill().bfill()

df_reservoir_inflow_entsoe_hourly = (
    df_wide[df_wide.index.year == baseyear]
    .stack()
    .reset_index()
    .rename(columns={0: 'natural_inflow'})
)

df_reservoir_inflow_entsoe_hourly.head()

C:\Users\jonas\AppData\Local\Temp\ipykernel_5580\3045521161.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_wide = df_wide.sort_index().resample('H').ffill().bfill()


,date,country,natural_inflow
0,2024-01-01,AT,0.000000
1,2024-01-01,BG,77.650238
2,2024-01-01,CH,0.000000
3,2024-01-01,ES,2176.482143
4,2024-01-01,FR,152.398810


In [73]:
# Upsample weekly reservoir inflows to hourly values (fixed)
df_reservoir_inflow_entsoe = df_reservoir_level_weekly_merged[['date','country','natural_inflow']].copy()

# replace negative inflows with 0 and convert to hourly rate
df_reservoir_inflow_entsoe['natural_inflow'] = df_reservoir_inflow_entsoe['natural_inflow'].clip(lower=0)
df_reservoir_inflow_entsoe['natural_inflow'] = df_reservoir_inflow_entsoe['natural_inflow'] / (7 * 24)

# add boundary timestamps for all countries so resampling can fill the full range
extras = []
start_ts = pd.Timestamp(f'{baseyear-1}-12-31')
end_ts = pd.Timestamp(f'{baseyear+1}-01-01')
for c in countries:
    extras.append({'date': start_ts, 'country': c, 'natural_inflow': np.nan})
    extras.append({'date': end_ts, 'country': c, 'natural_inflow': np.nan})
if extras:
    df_reservoir_inflow_entsoe = pd.concat([df_reservoir_inflow_entsoe, pd.DataFrame(extras)], ignore_index=True)

# pivot to wide, resample hourly and forward/back fill, then convert back to long
df_wide = df_reservoir_inflow_entsoe.set_index('date').pivot(columns='country', values='natural_inflow')
df_wide = df_wide.sort_index().resample('h').ffill().bfill()

df_reservoir_inflow_entsoe_hourly = (
    df_wide[df_wide.index.year == baseyear]
    .stack()
    .reset_index()
    .rename(columns={0: 'natural_inflow'})
)

df_reservoir_inflow_entsoe_hourly.head()

,date,country,natural_inflow
0,2024-01-01,AT,0.000000
1,2024-01-01,BG,77.650238
2,2024-01-01,CH,0.000000
3,2024-01-01,ES,2176.482143
4,2024-01-01,FR,152.398810


In [74]:
#Now distribute the inflows to storage technologies (PumpOpen and Reservoir) by their capacity
df_reservoir = pd.read_csv(fn_reservoir_size)
df_reservoir = df_reservoir.rename(columns={
    'Pump Storage - Closed Loop - Cumulated (upper or head) reservoir capacity (GWh)':'PumpClosed',
    'Pump Storage - Open Loop - Cumulated (upper or head) reservoir capacity (GWh)':'PumpOpen',
    'Reservoir - Reservoir capacity (GWh)':'Reservoir',
    'Run-of-River and pondage - Reservoir capacity linked to Run of River and Pondage units (GWh)':'RunOfRiver'
                                }).set_index('country')
df_reservoir = df_reservoir[['PumpClosed','PumpOpen','Reservoir','RunOfRiver']]
#temporary delete run of river from next equation: + df_reservoir['RunOfRiver']
df_reservoir_inflow = df_reservoir.copy().drop(columns={'RunOfRiver','PumpClosed'})
df_reservoir_inflow['sum'] = df_reservoir_inflow['PumpOpen'] + df_reservoir_inflow['Reservoir']
df_reservoir_inflow = pd.DataFrame(df_reservoir_inflow.stack()).reset_index()
df_reservoir_inflow.columns = ['country','technology','capacity_GWh']
df_reservoir_inflow['capacity_MWh'] = df_reservoir_inflow['capacity_GWh']*1000
df_reservoir_inflow = df_reservoir_inflow.pivot(index = 'country', columns = 'technology', values='capacity_MWh')
df_reservoir_inflow.head()

technology,PumpOpen,Reservoir,sum
country,,,
AL,0.0,1450000.0,1450000.0
AT,1722178.0,762386.2,2484564.2
BA,3400.0,1669000.0,1672400.0
BE,0.0,0.0,0.0
BG,255300.0,843000.0,1098300.0


In [75]:
#Calculate shares per technology
df_reservoir_inflow_shares = df_reservoir_inflow.copy()
df_reservoir_inflow_shares['PumpOpen'] = df_reservoir_inflow_shares['PumpOpen'] / df_reservoir_inflow_shares['sum']
df_reservoir_inflow_shares['Reservoir'] = df_reservoir_inflow_shares['Reservoir'] / df_reservoir_inflow_shares['sum']
df_reservoir_inflow_shares = df_reservoir_inflow_shares[['PumpOpen','Reservoir']]
df_reservoir_inflow_shares.head()

technology,PumpOpen,Reservoir
country,,
AL,0.000000,1.000000
AT,0.693151,0.306849
BA,0.002033,0.997967
BE,NaN,NaN
BG,0.232450,0.767550


In [76]:
#Now we distribute the inflows to storage technologies based on their shares
df_storage_inflows = df_reservoir_inflow_entsoe_hourly.copy()
df_storage_inflows = df_storage_inflows.merge(df_reservoir_inflow_shares,
                                                                how='left',on='country')   
df_storage_inflows['PumpOpen'] = df_storage_inflows['natural_inflow'] * df_storage_inflows['PumpOpen']
df_storage_inflows['Reservoir'] = df_storage_inflows['natural_inflow'] * df_storage_inflows['Reservoir']
df_storage_inflows = df_storage_inflows.melt(id_vars=['date','country'], value_vars=['PumpOpen','Reservoir'],
                                            var_name='technology', value_name='MWh')
df_storage_inflows = df_storage_inflows[['date','country','MWh','technology']]
df_storage_inflows.head() 

,date,country,MWh,technology
0,2024-01-01,AT,0.000000,PumpOpen
1,2024-01-01,BG,18.049810,PumpOpen
2,2024-01-01,CH,0.000000,PumpOpen
3,2024-01-01,ES,746.826189,PumpOpen
4,2024-01-01,FR,1.359355,PumpOpen


## Reservoir levels
Get hourly reservoir levels from weekly levels - to be used as storage start and end conditions

first, we load reservoir size to be able to distribute value by technology

In [77]:
df_reservoir = df_reservoir.drop(columns={'RunOfRiver'})
df_reservoir['sum'] = df_reservoir['PumpClosed'] + df_reservoir['PumpOpen'] + df_reservoir['Reservoir']
df_reservoir = pd.DataFrame(df_reservoir.stack()).reset_index()
df_reservoir.columns = ['country','technology','capacity_GWh']
df_reservoir['capacity_MWh'] = df_reservoir['capacity_GWh']*1000
df_reservoir = df_reservoir.pivot(index = 'country', columns = 'technology', values='capacity_MWh')
df_reservoir.head(1)

technology,PumpClosed,PumpOpen,Reservoir,sum
country,,,,
AL,0.0,0.0,1450000.0,1450000.0


replace absolute values with shares

In [78]:
df_reservoir_shares = df_reservoir.copy()
df_reservoir_shares['PumpClosed']  = df_reservoir_shares['PumpClosed'] / df_reservoir_shares['sum']
df_reservoir_shares['PumpOpen'] = df_reservoir_shares['PumpOpen'] / df_reservoir_shares['sum']
df_reservoir_shares['Reservoir'] = df_reservoir_shares['Reservoir'] / df_reservoir_shares['sum']
#df_reservoir_shares['RunOfRiver'] = df_reservoir_shares['RunOfRiver'] / df_reservoir_shares['sum']
df_reservoir_shares = df_reservoir_shares[['PumpClosed','PumpOpen','Reservoir']]
df_reservoir_shares.head()

technology,PumpClosed,PumpOpen,Reservoir
country,,,
AL,0.000000,0.000000,1.000000
AT,0.000000,0.693151,0.306849
BA,0.000000,0.002033,0.997967
BE,1.000000,0.000000,0.000000
BG,0.008486,0.230478,0.761036


and merge them with the storage sizes per technology

In [79]:
df_reservoir_level_merged = df_reservoir_level_weekly.reset_index().merge(df_reservoir_shares.reset_index(),
                                                                  how='left',left_on='country',
                                                                  right_on='country')
df_reservoir_level_merged['PumpClosed']  = df_reservoir_level_merged['PumpClosed'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged['PumpOpen'] = df_reservoir_level_merged['PumpOpen'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged['Reservoir'] = df_reservoir_level_merged['Reservoir'] * df_reservoir_level_merged['MWh']
#df_reservoir_level_merged['RunOfRiver'] = df_reservoir_level_merged['RunOfRiver'] * df_reservoir_level_merged['MWh']
df_reservoir_level_merged = pd.DataFrame(df_reservoir_level_merged.set_index(['date','country'])[
    ['PumpClosed','PumpOpen','Reservoir',]].stack()).reset_index().rename(
    columns={'level_2':'technology',0:'MWh'}).set_index(['date','country','technology'])
df_reservoir_level_merged.head(1)

,,,MWh
date,country,technology,
2024-01-01,RS,PumpClosed,0.0


In [80]:
#eventually, we create a df that has both relative level in % and absolute value in MWh

In [81]:
df_reservoir_level_final = df_reservoir_level_merged.reset_index().merge(
    pd.DataFrame(df_reservoir[['PumpClosed','PumpOpen','Reservoir']].stack()).rename(columns={0:'size_MWh'}).reset_index(),
    how='outer',on=['country','technology'])
df_reservoir_level_final['level'] = df_reservoir_level_final['MWh'] / df_reservoir_level_final['size_MWh']
df_reservoir_level_final = df_reservoir_level_final.fillna(0)
df_reservoir_level_final = df_reservoir_level_final[df_reservoir_level_final.country.isin(countries)]
df_reservoir_level_final.head()

,date,country,technology,MWh,size_MWh,level
108,2024-01-01 00:00:00,AT,PumpClosed,0.0,0.0,0.0
109,2024-01-08 00:00:00,AT,PumpClosed,0.0,0.0,0.0
110,2024-01-15 00:00:00,AT,PumpClosed,0.0,0.0,0.0
111,2024-01-22 00:00:00,AT,PumpClosed,0.0,0.0,0.0
112,2024-01-29 00:00:00,AT,PumpClosed,0.0,0.0,0.0


In [82]:
#check which countries do not have values assigned
missing = np.setdiff1d(countries,np.unique(df_reservoir_level_final.reset_index().country.unique()))
missing
#this looks ok

array(['DK', 'NL'], dtype='<U2')

## Reservoir size

we use the previous df, but in some cases, level is greater than size so we increase size to max level

In [83]:
df_reservoir_size = df_reservoir_level_final.drop(columns={'date'}).groupby(['country','technology']).max()
mask = df_reservoir_size['MWh'] > df_reservoir_size['size_MWh']
df_reservoir_size.loc[mask, 'size_MWh'] = df_reservoir_size.loc[mask, 'MWh']
df_reservoir_size = df_reservoir_size[['size_MWh']].rename(columns={'size_MWh':'MWh'})
df_reservoir_size.head()

MWh
country technology           
AT      PumpClosed        0.0
        PumpOpen    1722178.0
        Reservoir    762386.2
BE      PumpClosed     5300.0
        PumpOpen          0.0

## Net transfer capacity

Get net transfer capacities and select countries

In [84]:
df_ntc_in = pd.read_csv(fn_ntc, parse_dates=True, index_col="date").reset_index()
df_ntc  = df_ntc_in[
    (df_ntc_in["from"].isin(countries)) &
    (df_ntc_in["to"].isin(countries)) &
    (df_ntc_in["date"].dt.year == 2024)
]
df_ntc.head()

,date,from,to,ntc
0,2024-01-01,AT,CH,1200.0
1,2024-01-01,AT,CZ,900.0
2,2024-01-01,AT,DE,7500.0
3,2024-01-01,AT,HU,800.0
4,2024-01-01,AT,IT,875.0


Check if all countries have NTC values assigned

In [85]:
missing = np.setdiff1d(countries,np.unique(df_ntc[['to','from']]))
print(missing)

[]


## Hourly day ahead trade data

In [86]:
df_trade = pd.read_csv(fn_trade, parse_dates=True, index_col="time")
df_trade  = df_trade[(df_trade["from_country"].isin(countries))
                     & (df_trade["to_country"].isin(countries))]
df_trade.head()

,from_country,to_country,MWh
time,,,
2024-01-01 00:00:00,AT,CH,845.25
2024-01-01 01:00:00,AT,CH,1016.45
2024-01-01 02:00:00,AT,CH,1000.00
2024-01-01 03:00:00,AT,CH,1023.05
2024-01-01 04:00:00,AT,CH,990.80


In [87]:
missing = np.setdiff1d(countries,np.unique(df_trade['from_country']))
print(missing)

[]


## Prices

Get prices and select countries. 

In [88]:
df_price_in = pd.read_csv(fn_price, parse_dates=True, index_col="date").copy()
df_price_in = df_price_in.reindex(columns=countries).copy()
df_price_in.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8859 entries, 2024-01-01 00:00:00 to 2024-12-31 23:45:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8784 non-null   float64
 1   BE      8784 non-null   float64
 2   BG      8784 non-null   float64
 3   HR      8784 non-null   float64
 4   CZ      8784 non-null   float64
 5   DK      8784 non-null   float64
 6   FI      8784 non-null   float64
 7   FR      8784 non-null   float64
 8   DE      8784 non-null   float64
 9   GR      8784 non-null   float64
 10  HU      8784 non-null   float64
 11  IE      0 non-null      float64
 12  IT      8787 non-null   float64
 13  LU      8784 non-null   float64
 14  NL      8784 non-null   float64
 15  PL      8784 non-null   float64
 16  PT      8784 non-null   float64
 17  RO      8784 non-null   float64
 18  SK      8784 non-null   float64
 19  SI      8784 non-null   float64
 20  ES      8784 non-null   float64
 21  S

** NOTE: Here be careful about missing values. Correction should be done by hand.**

So we have a missing day for Switzerland, one for Ireland, and four hours for Poland. We assign values from Germany to Switzerland and Poland. And GB values to Ireland: 

In [89]:
df_price = df_price_in.copy()
df_price.CH = df_price.CH.fillna(df_price_in.DE)
df_price.PL = df_price.PL.fillna(df_price_in.DE)
df_price.IE = df_price.IE.fillna(df_price_in.GB)
df_price.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8859 entries, 2024-01-01 00:00:00 to 2024-12-31 23:45:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8784 non-null   float64
 1   BE      8784 non-null   float64
 2   BG      8784 non-null   float64
 3   HR      8784 non-null   float64
 4   CZ      8784 non-null   float64
 5   DK      8784 non-null   float64
 6   FI      8784 non-null   float64
 7   FR      8784 non-null   float64
 8   DE      8784 non-null   float64
 9   GR      8784 non-null   float64
 10  HU      8784 non-null   float64
 11  IE      0 non-null      float64
 12  IT      8787 non-null   float64
 13  LU      8784 non-null   float64
 14  NL      8784 non-null   float64
 15  PL      8784 non-null   float64
 16  PT      8784 non-null   float64
 17  RO      8784 non-null   float64
 18  SK      8784 non-null   float64
 19  SI      8784 non-null   float64
 20  ES      8784 non-null   float64
 21  S

Furthermore, if we add EU-27 countries, HR and RO are missing values
    using SI for HR (seems to be the neighbour with most similar prices)
    and BG for RO
    (used this as rough proxy: https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:52019DC0001&from=EN)

In [90]:
df_price = df_price.copy()
df_price.HR = df_price.HR.fillna(df_price_in.SI)
df_price.RO = df_price.RO.fillna(df_price_in.BG)
df_price.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8859 entries, 2024-01-01 00:00:00 to 2024-12-31 23:45:00
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AT      8784 non-null   float64
 1   BE      8784 non-null   float64
 2   BG      8784 non-null   float64
 3   HR      8784 non-null   float64
 4   CZ      8784 non-null   float64
 5   DK      8784 non-null   float64
 6   FI      8784 non-null   float64
 7   FR      8784 non-null   float64
 8   DE      8784 non-null   float64
 9   GR      8784 non-null   float64
 10  HU      8784 non-null   float64
 11  IE      0 non-null      float64
 12  IT      8787 non-null   float64
 13  LU      8784 non-null   float64
 14  NL      8784 non-null   float64
 15  PL      8784 non-null   float64
 16  PT      8784 non-null   float64
 17  RO      8784 non-null   float64
 18  SK      8784 non-null   float64
 19  SI      8784 non-null   float64
 20  ES      8784 non-null   float64
 21  S

## Cost Resource Curves

In [91]:
df_cost_res_in = pd.read_csv(fn_cost_res) 

In [92]:
df_cost_res = df_cost_res_in[(df_cost_res_in.country.isin(countries))
                  & df_cost_res_in.technology.isin(technologies)].copy()

Add plant index to dataframe

In [93]:
df_cost_res["plant"] = df_cost_res.country + "_" + df_cost_res.technology
df_cost_res.head(1)

,country,technology,cinv_0,cinv_1,potential_twh,potential_mw,plant
0,AT,Solar,36.213937,2.073166e-08,86.530179,79833.267526,AT_Solar


check for missing countries in df_cost_res

In [95]:
missing = np.setdiff1d(countries,np.unique(df_cost_res.country.unique()))
missing

array([], dtype='<U2')

## CHP Data

In [96]:
df_chp_gen_in = pd.read_csv(fn_chp_gen)
df_chp_gen_in.loc[df_chp_gen_in['year'] == 2017, 'year'] = 2024
df_chp_gen_in.head(1)

,year,country,tech,MWh
0,2015,BE,Biomass,525738.824874


Select year and countries:

In [97]:
df_chp_gen = df_chp_gen_in[(df_chp_gen_in.year == baseyear)
                          & (df_chp_gen_in.country.isin(countries))].drop("year", axis = 1).rename(
columns={'tech':'technology'})
df_chp_gen.head()

,country,technology,MWh
58,BE,Biomass,4.764971e+05
59,BG,Biomass,2.750572e+04
60,CZ,Biomass,2.168299e+05
61,DK,Biomass,3.682033e+06
62,DE,Biomass,9.952557e+06


check for missing countries in df_chp_gen

In [98]:
missing = np.setdiff1d(countries,np.unique(df_chp_gen.country.unique()))
missing
#only CH missing from eurostat data - CHP not relevant for Nuclear since baseload

array(['CH'], dtype='<U2')

Also get profiles and add a date for the current baseyear

In [99]:
df_heat_dem_in = pd.read_csv(fn_heat_dem)
dates = pd.date_range(start='%d-01-01 00:00:00' % baseyear, 
                      end='%d-12-31 23:00:00' % baseyear, 
                      periods = len(df_heat_dem_in))
df_heat_dem_in = df_heat_dem_in.set_index(dates)
df_heat_dem_in.index.name = "date"
df_heat_dem_in = df_heat_dem_in.reset_index()
df_heat_dem_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  8760 non-null   datetime64[ns]
 1   heat_demand           8760 non-null   float64       
 2   heat_demand_relative  8760 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 205.4 KB


# Upward adjustment of capacities if hourly entsoe value is higher

In [185]:
df_hourly_entsoe_max = df_hourly_entsoe.reset_index().groupby(['country','technology']).max()
df_hourly_entsoe_max = df_hourly_entsoe_max.reset_index().drop(columns={'time'})
df_hourly_entsoe_max = df_hourly_entsoe_max[(df_hourly_entsoe_max.country.isin(countries))
                  & df_hourly_entsoe_max.technology.isin(technologies)]
df_hourly_entsoe_max.head()

,country,technology,MWh_hourly
0,AT,Biomass,312.00
1,AT,Gas,3731.20
2,AT,HardCoal,0.00
3,AT,Oil,0.00
4,AT,Other,122.07


In [186]:
df_cap_test = df_cap.copy().reset_index()
df_cap_test = df_cap_test.merge(df_hourly_entsoe_max,left_on=['country','technology'],right_on=['country','technology'],how='left')
df_cap_test.head()

,technology,country,capacity,pumping,MWh_hourly
0,Biomass,AT,549.0,0.0,312.0
1,Gas,AT,4225.0,0.0,3731.2
2,HardCoal,AT,0.0,0.0,0.0
3,Lignite,AT,0.0,0.0,NaN
4,Nuclear,AT,0.0,0.0,NaN


In [187]:
#df_cap_test[df_cap_test.MWh_hourly > df_cap_test.capacity]

In [188]:
df_cap_test.capacity[df_cap_test.MWh_hourly > df_cap_test.capacity] = df_cap_test.MWh_hourly

C:\Users\jonas\AppData\Local\Temp\ipykernel_5580\216730869.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_cap_test.capacity[df_cap_test.MWh_hourly > df_cap_test.capacity] = df_cap_test.MWh_hourly
C:\Users\jonas\AppData\Local\Temp\ipy

# GDX export

In [189]:
# 1. Create the basic mapping
df_periods = pd.Series({
    d: "t%04d" % (i+1) for i, d in enumerate(df_load.reset_index().time.unique())
}).to_frame("period")

df_periods.index = pd.to_datetime(df_periods.index)

if df_periods.index.tz is None:
    df_periods.index = df_periods.index.tz_localize('UTC')
else:
    df_periods.index = df_periods.index.tz_convert('UTC')
# -----------------------------------------------

# 2. Add the helper columns
df_periods.index.name = "time"
df_periods['day'] = df_periods.index.date
df_periods['month'] = df_periods.index.month
df_periods["periodLength"] = 1

# 3. Filter and Reset
# Note: Filtering Leap Year rows
df_periods = df_periods[~((df_periods.index.month == 2) & (df_periods.index.day == 29))]
df_periods = df_periods.reset_index()

In [190]:
import gams.transfer as gt
m = gt.Container()

c:\Users\jonas\anaconda3\Lib\site-packages\gams\transfer\containers\_container.py:72: UserWarning: The GAMS version (44.4.0) differs from the API version (53.3.0).
  ws = GamsWorkspace()


In [191]:
# --- 1. SETS ---
# Simple sets (1D)
# We ensure fuels and technologies are strings and not null
fuels_clean = [str(f) for f in fuels if pd.notnull(f)]
tech_clean = [str(t) for t in technologies if pd.notnull(t)]

c = m.addSet("c", records=sorted(countries), description="countries")
eu = m.addSet("eu", records=sorted(countries_EU_27), description="EU 27 countries")
tech = m.addSet("tech", records=tech_clean, description="technologies")
i = m.addSet("i", records=conventionals, description="conventional technologies")
s = m.addSet("s", records=storages, description="storage facilities")
r = m.addSet("r", records=renewables, description="new renewable sources")
m.addSet("ro", records=old_renewables, description="conventional renewables")
m.addSet("ra", records=all_renewables, description="all renewable sources")
m.addSet("baseload", records=baseload, description="baseload excl RoR")
m.addSet("fixed", records=fixed_feedin, description="fixed feedin tech")
m.addSet("peak", records=peakload, description="peak load technologies")
f = m.addSet("f", records=fuels_clean, description="fuels")

#Add aliases for sets
m.addAlias("cc", c)

# Mappings (Multi-dimensional sets)
# FIX: dropna() ensures no "Categorical categories cannot be null" error
m.addSet("mapTF", domain=["tech", "f"], 
        records=df_cost_in[["Technology", "Main Fuel"]].dropna(), 
        description="mapping technology to fuel")
    
m.addSet("map_fuel_price_as", domain=["c", "cc", "f"], 
         records=fuel_price_maps, description="mapping country to fuel prices")

m.addSet("map_om_cost_as", domain=["c", "cc", "tech"], 
         records=om_maps, description="mapping country to tech for OM costs")

# --- 2. COUNTRY PARAMETERS ---
m.addParameter("penalty", domain=["c"], records=df_countries.UpperPriceLimit)
m.addParameter("curtPenalty", domain=["c"], records=df_countries.CurtailmentPenalty)
m.addParameter("reserveRequirement", domain=["c"], records=df_countries.ReserveRequirement)
m.addParameter("epsilon", domain=["c"], records=df_countries.DemandElasticity)
m.addParameter("p_carb", domain=["c"], records=df_countries.CarbonPrice)

# --- 3. PLANT AND COST SPECIFICATIONS ---
df_c = df_cost.set_index(["Technology", "country"])

m.addParameter("eta", domain=["tech", "c"], records=df_c["Average Efficiency"])
m.addParameter("availUp", domain=["tech", "c"], records=df_c["Availability"])
m.addParameter("min_gen", domain=["tech", "c"], records=df_c["Min Generation"])
m.addParameter("loadGradient", domain=["tech", "c"], records=df_c["LoadGradient"])
m.addParameter("rampingCost", domain=["tech", "c"], records=df_c["RampingCost"])
m.addParameter("rampingCostII", domain=["tech", "c"], records=df_c["RampingCostII"])

# --- 4. CAPACITIES AND RESERVOIRS ---
m.addParameter("cap", domain=["tech", "c"], 
               records=df_cap.capacity, 
               description="installed capacity [MW]")

m.addParameter("cap_pump", domain=["tech", "c"], 
               records=df_cap.pumping, 
               description="pumping capacity [MW]")

m.addParameter("reservoir_size_up", domain=["tech", "c"], 
               records=df_reservoir_size.MWh, 
               description="reservoir size [MWh]")

# --- 5. RENEWABLE COST CURVES ---
df_cr = df_cost_res.set_index(["technology", "country"])
m.addParameter("cinv_0", domain=["tech", "c"], records=df_cr["cinv_0"])
m.addParameter("cinv_1", domain=["tech", "c"], records=df_cr["cinv_1"])
m.addParameter("pot_ren_mwh", domain=["tech", "c"], records=df_cr["potential_twh"] * 1e6)

# --- 6. FUEL SPECIFICATIONS ---
m.addParameter("carb_coef", domain=["f"], records=df_fuels[df_fuels["Main Fuel"].isin(fuels_clean)].set_index("Main Fuel")["Carbon"])
# Fuel Prices (pf and pf_2)
df_f2017 = df_fuels_2017.set_index(["Main Fuel", "country"])[["Price", "Price_2"]].dropna()
m.addParameter("pf", domain=["f", "c"], records=df_f2017.Price)
m.addParameter("pf_2", domain=["f", "c"], records=df_f2017.Price_2)
# Variable O&M
m.addParameter("c_vom", domain=["tech", "c"], 
               records=df_OM_2017.fillna(0).set_index(["Technology", "country"]).variable_OM_Cost)
# Yearly Generation / CHP
m.addParameter("chp_gen", domain=["tech", "c"], 
               records=df_chp_gen.fillna(0).set_index(["technology", "country"]).MWh)

m.addParameter("gen_annual", domain=["c", "tech"], 
               records=df_gen_annual.MWh)

<Parameter `gen_annual` (0x1f734f4be90)>

In [192]:
# 1. SETS
m.addSet("t", records=df_periods.period.unique(), description="Periods")
m.addSet("d", records=df_periods.day.unique(), description="days")
m.addSet("m", records=df_periods.month.unique(), description="months")

# Mapping Sets (Tuples)
m.addSet("map_t_d", domain=["t", "d"], records=df_periods[["period", "day"]])
m.addSet("map_t_m", domain=["t", "m"], records=df_periods[["period", "month"]])

# 2. PARAMETERS

# Period Duration
m.addParameter("dur_d", domain=["t"], 
               records=df_periods.set_index("period")["periodLength"])
# Demand
df_d = df_load.reset_index().copy()
df_d['time'] = df_d['time'].dt.tz_localize('UTC')
df_d = df_d.merge(df_periods, on="time", how="left")
df_d = df_d[["country", "period", "MWh"]].copy()
df_d = df_d.dropna(subset=["period"]).set_index(["country", "period"])
m.addParameter("demand", domain=["country", "t"], 
               records=df_d["MWh"])

# Reference Prices
df_p = df_price.unstack().reset_index()
df_p.columns = ["country", "date", "price"]
df_p['date'] = df_p['date'].dt.tz_localize('UTC')
df_p = df_p.merge(df_periods, left_on="date", right_on="time", how="left")
df_p = df_p[["country", "period", "price"]].copy()
df_p = df_p.dropna(subset=["period"]).set_index(["country", "period"])
m.addParameter("pRef", domain=["c", "t"], 
               records=df_p["price"].fillna(0))

# Monthly Generation & Availability
m.addParameter("gen_monthly", domain=["m", "c", "tech"], 
               records=df_generation_monthly["MWh"])

m.addParameter("availpeakUp", domain=["c", "tech", "m"], 
               records=df_avail_peak["avail"])

# Renewable Supply (renS)
df_r = df_res.copy().reset_index()
df_r['time'] = df_r['time'].astype('datetime64[ns, UTC]')
df_r['time'] = pd.to_datetime(df_r['time'], utc=True, errors='coerce')
df_r = df_r.merge(df_periods, left_on="time", right_on="time", how="left")
df_r = df_r[["country", "technology", "period", "profile"]].copy()
df_r = df_r.dropna(subset=["period"]).set_index(["country", "technology", "period"])
df_r = df_r[df_r.index.get_level_values('country').isin(countries)]
#here we drop duplicates. currently do not know why we have them, but we need to drop them to avoid errors in GAMS. We keep the first value, but it should not matter as they are duplicates.
df_r = df_r[~df_r.index.duplicated(keep=False)]
m.addParameter("renS", domain=["c", "tech", "t"],
               records=df_r["profile"])

# NTC Values
df_n = df_ntc.copy()
df_n['date'] = df_n['date'].dt.tz_localize('UTC')
df_n = df_n.merge(df_periods, left_on="date", right_on="time", how="left")
df_n = df_n[["from", "to", "period", "ntc"]]
df_n = df_n.dropna(subset=["period"]).set_index(["from", "to", "period"])
df_n.head()
m.addParameter("ntc", domain=["c", "cc", "t"], 
               records=df_n["ntc"])

# Hydro / Run-of-River
df_ro = df_ror.copy()
df_ro['time'] = pd.to_datetime(df_ro['time'], utc=True, errors='coerce')
df_ro = df_ro.merge(df_periods, left_on="time", right_on="time", how="left")
df_ro = df_ro[['country','period','MWh']]
df_ro = df_ro.dropna(subset=["period"]).set_index(["country", "period"])
m.addParameter("gen_ror_exog", domain=["c", "t"], 
                   records=df_ro['MWh'])

# Reservoirs (Initial Levels and Inflows)
df_re = df_reservoir_level_final.copy()
df_re['date'] = pd.to_datetime(df_re['date'], utc=True, errors='coerce')
df_re = df_re.merge(df_periods, left_on="date", right_on="time", how="left")
df_re = df_re[["country", "technology", "period", "MWh"]]
df_re = df_re.dropna(subset=["period"]).set_index(["country", "technology", "period"])
df_re = df_re[df_re.index.get_level_values('country').isin(countries)]
m.addParameter("reservoir_initial_up", domain=["c", "tech", "t"],   
               records=df_re["MWh"])                   

df_sto = df_storage_inflows.copy()
df_sto['date'] = df_sto['date'].dt.tz_localize('UTC')
df_sto = df_sto.merge(df_periods, left_on="date", right_on="time", how="left")
df_sto = df_sto[["country", "technology", "period", "MWh"]]
df_sto = df_sto.dropna(subset=["period"]).set_index(["country", "period", "technology"])
df_sto = df_sto[df_sto.index.get_level_values('country').isin(countries)]
m.addParameter("reservoir_inflow_up", domain=["c", "t", "tech"], 
                   records=df_sto["MWh"].fillna(0))

df_h = df_heat_dem_in.copy()
df_h['date'] = df_h['date'].dt.tz_localize('UTC').dt.floor('h')
df_h = df_h.merge(df_periods, left_on="date", right_on="time", how="left")
df_h = df_h[["period", "heat_demand_relative"]]
df_h = df_h.dropna(subset=["period"]).set_index("period")
m.addParameter("heat_dem_up", domain=["t"], 
               records=df_h["heat_demand_relative"])



<Parameter `heat_dem_up` (0x1f734deb2d0)>

In [193]:
#Export to GDX
fn_out = os.path.join(dir_gdx, f"data{countries_out}_{baseyear}_all.gdx")

m.write(fn_out)

print(f"GDX successfully written to: {fn_out}")


GDX successfully written to: c:\Users\jonas\Documents\eu_electricity_model\master\model\data\data_EU_2024_all.gdx


# @Jonas CONTINUE FROM HERE

### Calibration data

Static data plus the following dynamic parameters:
- r_price(country,t)
- r_demand(country,t)
- r_generation(tech,country,t)
- TRADE(fromcountry,tocountry,t)

In [ ]:
def inject_static_calibration(gdx):
    """Injects static data into gdx file
    :param gdx: <gams.GamsDatabase> gdx to inject data
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # sets
    ## technology sets
    gt.add_set(sorted(countries), "c", gdx, text="countries", overwrite=overwrite)
    gt.add_set(technologies, "tech", gdx, text="technologies", overwrite=overwrite)
    gt.add_set(conventionals, "i", gdx, text="conventional technologies", overwrite=overwrite)
    gt.add_set(storages, "s", gdx, text="storage facilities", overwrite=overwrite)  
    gt.add_set(renewables, "r", gdx, text="new renewable sources", overwrite=overwrite)  
    gt.add_set(old_renewables, "ro", gdx, text="old = conventional renewable sources", overwrite=overwrite)  
    gt.add_set(all_renewables, "ra", gdx, text="all renewable sources", overwrite=overwrite)  
    gt.add_set(baseload, "baseload", gdx, text="baseload technologies excl. RoR", overwrite=overwrite)  
    gt.add_set(fixed_feedin, "fixed", gdx, text="fixed feedin technologies", overwrite=overwrite)  
    gt.add_set(peakload, "peak", gdx, text="peak load technologies", overwrite=overwrite)  


In [ ]:
def inject_dynamic_calibration(gdx, df_period):
    """ Injects time dependent parameters into gdx given period selection
    :param gdx: <gams.GamsDatabase> gdx to inject data
    :param df_periods: <pd.DataFrame> with mapping from dates to periods and lenght of respective period
    """
    # overwrite settings (False does not work due to bug in gdxtools)
    overwrite = True
    
    # extract mapping dates to periods
    map_periods = df_periods.period.to_dict()
    periods = map_periods.values()
    
    # sets
    gt.add_set(df_periods.period, "t", gdx, text="Periods", overwrite=overwrite)
    gt.add_set(df_periods.day, "d", gdx, text="days", overwrite=overwrite)
    gt.add_set(df_periods.month, "m", gdx, text="months", overwrite=overwrite)
    
    # period duration
    gt.add_parameter(df_period.reset_index().set_index("period").periodLength.to_dict(), "dur_d", gdx,
                  text="duration of days", overwrite=overwrite)
    
    # day mapping
    gt.add_set(list(df_periods.reset_index()[["period", "day"]].itertuples(index=False, name=None)),
               "map_t_d", gdx, text="mapping of periods to dates",
               overwrite=overwrite)
    
    # month mapping
    gt.add_set(list(df_periods.reset_index()[["period", "month"]].itertuples(index=False, name=None)),
               "map_t_m", gdx, text="mapping of periods to months",
               overwrite=overwrite)
    
    # extract and select time related parameters
    def extract_and_select(df):
        df_ = df.unstack().reset_index()
        df_["period"] = df_.date.map(map_periods)
        df_ = df_[df_.period.notnull()] 
        return df_
    
    # prices
    df_ = extract_and_select(df_price)
    gt.add_parameter(df_.set_index(["level_0","period"])[0].to_dict(), "r_price", gdx,
                text="hourly reference price [Euro per MWh]", overwrite=overwrite) 
    
    # load 
    df_ = df_load.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "r_demand", gdx,
                text="demand per period [MWh]", overwrite=overwrite)  
    
    # generation
    df_ = df_generation.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["technology", "country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "r_generation", gdx,
                text="hourly electricity generation [MWh]", overwrite=overwrite)  
    
    # trade
    df_ = df_trade.reset_index()
    df_["period"] = df_.time.map(map_periods)
    df_ = df_[df_.period.notnull()]
    df_ = df_.set_index(["from_country", "to_country", "period"])
    gt.add_parameter(df_.MWh.to_dict(), "TRADE", gdx,
                text="hourly cross-border trade [MWh]", overwrite=overwrite)   

### Short data set: One day per month


In [ ]:
# to be done in future